<a href="https://colab.research.google.com/github/felmar110/Chatbot-RAG-Asistente-para-la-gestion-del-mantenimiento-Crossorter/blob/main/Chatbot_crossorter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## RAG System with interfaz

## 1. Instalación y Configuración de Librerías
Este bloque de código instala y actualiza las librerías de Python necesarias para el funcionamiento del chatbot. Incluye herramientas para la manipulación de datos (`numpy`, `scipy`, `scikit-learn`, `pandas`), procesamiento de PDFs (`pypdf`), lectura de Excel (`openpyxl`), creación de interfaces (`gradio`), bases de datos vectoriales (`chromadb`), tokenización (`tiktoken`), y la API de Google Generative AI (`google-generativeai`). También realiza una verificación de diagnóstico para `gspread`.

In [ ]:
import sys

# Completely remove all explicit uninstall commands.
# Instead, rely on `pip install --upgrade` to update packages
# and resolve dependencies within the existing stable Colab environment.
# Forcing reinstallation of gspread, numpy, scipy, and scikit-learn due to persistent 'open_by_id' and numpy/scipy dependency issues.
!{sys.executable} -m pip install -q --upgrade --force-reinstall gspread numpy scipy scikit-learn pypdf openpyxl gradio chromadb tiktoken google-generativeai
!{sys.executable} -m pip install -q google-colab

print("Libraries installed successfully!")

# Small diagnostic check for gspread, using the correct authentication flow for Colab
try:
    import gspread
    from google.auth import default
    # Attempt to get default credentials
    creds, _ = default()
    # Attempt to create a dummy client to check if 'open_by_id' is available with these credentials
    dummy_gc = gspread.Client(creds)

    if hasattr(dummy_gc, 'open_by_id'):
        print("gspread.Client tiene el método 'open_by_id'. La instalación parece correcta.")
    else:
        print("ADVERTENCIA: gspread.Client NO tiene el método 'open_by_id'. Todavía podría haber un problema. Verificando la versión de gspread.")
        print(f"Versión de gspread instalada: {gspread.__version__}")
except Exception as e:
    print(f"Error durante la verificación de diagnóstico de gspread: {e}")

## 2. Autenticación con Google Sheets
Este paso es crucial para permitir que el Colab interactúe con los servicios de Google, como Google Sheets. La autenticación se realiza a través de la cuenta de usuario de Colab, obteniendo credenciales por defecto que `gspread` utiliza para autorizarse.

In [ ]:
from google.colab import auth
import gspread
from google.auth import default # Needed for user authentication

# Authenticate to Google Colab to access your Google Drive/Sheets
# This will open a browser window for you to grant permissions.
auth.authenticate_user()

# Get the default credentials provided by Colab's authentication
creds, _ = default()

# Authorize gspread to use the authenticated user's credentials
# Using gspread.authorize is the recommended way with existing credentials.
gc = gspread.authorize(creds)

print("Autenticación con Google Sheets completada.")
print("Ahora puedes usar 'gc' para abrir tus hojas de cálculo.")

Autenticación con Google Sheets completada.
Ahora puedes usar 'gc' para abrir tus hojas de cálculo.


## 3. Configuración de Google Generative AI (Gemini API)
Aquí se importa el SDK de Google Generative AI y se configura con la clave API obtenida de los secretos de Colab. Esta configuración es esencial para poder hacer llamadas a los modelos de lenguaje de Gemini para la generación de texto.

In [ ]:
# Import the Python SDK
import google.generativeai as genai

# Used to securely store your API key
from google.colab import userdata

# Get API key from Colab's secrets manager
GOOGLE_API_KEY = userdata.get('Gemini_API_Key')

# Configure the generative AI model
genai.configure(api_key=GOOGLE_API_KEY)

print("Google Generative AI configured successfully!")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Google Generative AI configured successfully!


## 4. Carga del Vector Store (si existe)
Este bloque intenta cargar un `vector_store` preexistente desde un archivo. Un vector store almacena los embeddings de los documentos procesados, lo que evita tener que regenerarlos cada vez que se ejecuta el notebook, ahorrando tiempo y uso de cuotas de API. Si no encuentra el archivo, inicializa el `vector_store` como vacío.

### Cargar el Vector Store
Para evitar re-ejecutar el procesamiento de documentos y la generación de embeddings, puedes cargar el `vector_store` guardado al inicio de tu notebook.

In [ ]:
# Cargar el vector_store si el archivo existe
try:
    if os.path.exists(VECTOR_STORE_FILENAME):
        with open(VECTOR_STORE_FILENAME, 'rb') as f:
            vector_store = pickle.load(f)
        print(f"Vector store cargado exitosamente desde '{VECTOR_STORE_FILENAME}'")
        # You might want to set a flag here to skip subsequent document processing if loaded successfully
        vector_store_loaded = True
    else:
        print("No se encontró un vector store guardado. Se procederá con la generación de embeddings.")
        vector_store = [] # Initialize as empty list if not loaded
        vector_store_loaded = False
except Exception as e:
    print(f"Error al cargar el vector store: {e}")
    vector_store = [] # Re-initialize if there was an error during loading
    vector_store_loaded = False

Vector store cargado exitosamente desde './data/vector_store.pkl'


## 5. Preparación del Directorio de Datos
Se define y crea un directorio llamado `./data` donde se espera que el usuario suba sus documentos (PDF y Excel). Esto organiza los archivos de origen que serán procesados por el chatbot.

In [ ]:
import os

# Create a directory for your documents if it doesn't exist
DATA_DIR = "./data"
if not os.path.exists(DATA_DIR):
    os.makedirs(DATA_DIR)

print(f"Please upload your PDF and Excel documents to the '{DATA_DIR}' folder.")
print("For demonstration, you can place a sample PDF named 'sample.pdf' and an Excel file named 'sample.xlsx' here.")

Please upload your PDF and Excel documents to the './data' folder.
For demonstration, you can place a sample PDF named 'sample.pdf' and an Excel file named 'sample.xlsx' here.


## 6. Definición del Nombre del Archivo del Vector Store
Se especifica el nombre y la ruta completa para el archivo donde se guardará o cargará el vector store (`vector_store.pkl`) dentro del directorio de datos. Esto asegura que el `vector_store` se maneje de forma consistente.

In [ ]:
import pickle

# Define the path for the saved vector store
VECTOR_STORE_FILENAME = os.path.join(DATA_DIR, "vector_store.pkl")

print(f"Vector store will be saved/loaded at: {VECTOR_STORE_FILENAME}")

Vector store will be saved/loaded at: ./data/vector_store.pkl


## 7. Carga y Preprocesamiento de Documentos (PDF y Excel)
Este código itera a través del directorio `./data` para cargar documentos PDF y Excel. Extrae texto de las páginas de los PDFs y convierte las hojas de cálculo de Excel a texto. Cada pieza de texto se almacena junto con metadatos (origen, página/hoja) en una lista `all_documents`.

In [ ]:
import pypdf # For PDF loading
import pandas as pd # For Excel loading
import os

# This list will store dictionaries, each representing a processed document or part of it
# Each dictionary will have 'text', 'source', and potentially other metadata
all_documents = []

# Load PDF documents
for file_name in os.listdir(DATA_DIR):
    if file_name.endswith(".pdf"):
        file_path = os.path.join(DATA_DIR, file_name)
        try:
            reader = pypdf.PdfReader(file_path)
            for i, page in enumerate(reader.pages):
                text = page.extract_text()
                if text:
                    all_documents.append({
                        'text': text,
                        'source': file_name,
                        'page': i + 1
                    })
            print(f"Loaded {len(reader.pages)} pages from {file_name}")
        except Exception as e:
            # Added more specific error handling for pypdf encoding issues
            print(f"Error loading PDF {file_name} (Page {i+1} if applicable): {e}")
            print(f"This might be due to unsupported PDF encoding or corruption. Skipping this PDF.")


# Load Excel documents
for file_name in os.listdir(DATA_DIR):
    if file_name.endswith(".xlsx"):
        file_path = os.path.join(DATA_DIR, file_name)
        try:
            xls = pd.ExcelFile(file_path)
            for sheet_name in xls.sheet_names:
                df = pd.read_excel(xls, sheet_name=sheet_name)
                # Convert DataFrame to a string representation for embedding
                text_content = df.to_string(index=False)
                all_documents.append({
                    'text': text_content,
                    'source': file_name,
                    'sheet': sheet_name
                })
            print(f"Loaded sheets from {file_name}")
        except Exception as e:
            print(f"Error loading Excel {file_name}: {e}")


print(f"Total documents (pages/sheets) loaded: {len(all_documents)}")
# Display the first document to see its structure
if all_documents:
    print("First loaded document (text snippet and metadata):")
    # Display first 500 characters of text for brevity
    display({k: v if k != 'text' else v[:500] + '...' for k, v in all_documents[0].items()})

Loaded 22 pages from Interfaz eDs – Host Coordinadora.pdf


ERROR:pypdf._cmap:Advanced encoding /SymbolSetEncoding not implemented yet
ERROR:pypdf._cmap:Advanced encoding /SymbolSetEncoding not implemented yet


Error loading PDF MANTENIMIENTO DEL CROSSORTER 1500.pdf (Page 1 if applicable): unknown encoding: /SymbolSetEncoding
This might be due to unsupported PDF encoding or corruption. Skipping this PDF.
Loaded 60 pages from DATOS VGI.pdf
Loaded 40 pages from 1408915 - COO-Manual conservación_ver2 .pdf
Total documents (pages/sheets) loaded: 122
First loaded document (text snippet and metadata):


{'text': ' \n Referencia: 08915-454-11000-ES-B03 - SW Design Description   \n \n \n \nPROYECTO Nº \n08915 \n \n \n \n \n \n \n \n \nSW Design Description \nInterfaz eDs – Host Coordinadora \n \n \n \n \n \nBarcelona, 14 de Septiembre de 2020 \n \nRev.B \n \n \n \n \n \n \nVanderlande Industries España, S.A.U.  C/Joan d’Àustria, 39 -47 2º 08005 – Barcelona, España \nN.I.F. A60582913  Tel. 93.221.94.94  Fax. 93.221.90.99   www.vanderlande.com \n \n...',
 'source': 'Interfaz eDs – Host Coordinadora.pdf',
 'page': 1}

## 8. División de Texto en Chunks (Fragmentación)
La función `custom_text_splitter` toma los documentos cargados y los divide en fragmentos (`chunks`) más pequeños de un tamaño específico (`chunk_size`) y con una superposición (`chunk_overlap`). Esto es fundamental para el RAG, ya que permite que el modelo procese porciones de texto manejables y capture el contexto relevante sin exceder los límites de tokens.

In [ ]:
def custom_text_splitter(documents, chunk_size=1000, chunk_overlap=200):
    chunks = []
    for doc in documents:
        text = doc['text']
        metadata = {k: v for k, v in doc.items() if k != 'text'}

        # Simple character-based splitting
        start = 0
        while start < len(text):
            end = start + chunk_size
            chunk_text = text[start:end]

            # Create a chunk dictionary
            chunk = {
                'text': chunk_text,
                'metadata': metadata
            }
            chunks.append(chunk)
            start += (chunk_size - chunk_overlap)
            # Ensure start doesn't go beyond text length if the last chunk is very small
            if start >= len(text) - chunk_overlap and len(text) - start < chunk_overlap and start < len(text):
                break # Avoid adding very tiny trailing chunks unnecessarily

    # Further refinement to ensure no empty chunks and proper overlap for the last chunk
    final_chunks = []
    for doc in documents:
        text = doc['text']
        metadata = {k: v for k, v in doc.items() if k != 'text'}

        start_idx = 0
        while start_idx < len(text):
            end_idx = min(start_idx + chunk_size, len(text))
            chunk_text = text[start_idx:end_idx]
            if chunk_text.strip(): # Only add non-empty chunks
                final_chunks.append({
                    'text': chunk_text,
                    'metadata': metadata
                })
            start_idx += (chunk_size - chunk_overlap)
            if start_idx > len(text): # Prevent infinite loops on very small texts
                break

    return final_chunks

# Initialize the text splitter (using our custom function)
chunks = custom_text_splitter(all_documents, chunk_size=1000, chunk_overlap=200)

print(f"Number of chunks created: {len(chunks)}")
# Display the first chunk
if chunks:
    print("First chunk (text snippet and metadata):")
    display({k: v if k != 'text' else v[:500] + '...' for k, v in chunks[0].items()})

Number of chunks created: 386
First chunk (text snippet and metadata):


{'text': ' \n Referencia: 08915-454-11000-ES-B03 - SW Design Description   \n \n \n \nPROYECTO Nº \n08915 \n \n \n \n \n \n \n \n \nSW Design Description \nInterfaz eDs – Host Coordinadora \n \n \n \n \n \nBarcelona, 14 de Septiembre de 2020 \n \nRev.B \n \n \n \n \n \n \nVanderlande Industries España, S.A.U.  C/Joan d’Àustria, 39 -47 2º 08005 – Barcelona, España \nN.I.F. A60582913  Tel. 93.221.94.94  Fax. 93.221.90.99   www.vanderlande.com \n \n...',
 'metadata': {'source': 'Interfaz eDs – Host Coordinadora.pdf', 'page': 1}}

## 9. Generación de Embeddings y Función de Recuperación (RAG)
Aquí se utiliza el modelo de embedding de Gemini (`gemini-embedding-2`) para convertir cada fragmento de texto en un vector numérico (embedding). Estos embeddings se almacenan en el `vector_store`. Además, se define la función `retrieve_documents` que, dada una consulta, genera su embedding y busca los fragmentos de documento más similares en el `vector_store` utilizando la similitud del coseno.

In [ ]:
import google.generativeai as genai

# Assuming your API key is already configured
# genai.configure(api_key=YOUR_API_KEY)

print("Available Embedding Models:")
for m in genai.list_models():
    if "embedContent" in m.supported_generation_methods:
        print(m.name)

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Ensure the Generative AI model is configured
# This assumes `genai` and `GOOGLE_API_KEY` are already configured from previous cells.
# Configure if not already done (though it should be by cell 9daa6034)
# genai.configure(api_key=GOOGLE_API_KEY)

# Initialize the embedding model directly using genai
# The task type is important for optimal performance in retrieval scenarios
embedding_model = "models/gemini-embedding-2" # Updated to the correct model name

# This list will store tuples of (embedding_vector, chunk_metadata)
vector_store = []

print("Generating embeddings for document chunks...")
for i, chunk in enumerate(chunks):
    try:
        # Directly use genai.embed_content
        embedding_response = genai.embed_content(
            model=embedding_model,
            content=chunk['text'],
            task_type="RETRIEVAL_DOCUMENT"
        )
        embedding_vector = embedding_response['embedding']
        vector_store.append((embedding_vector, chunk['metadata'], chunk['text']))
        if i % 10 == 0: # Print progress every 10 chunks
            print(f"Processed {i+1}/{len(chunks)} chunks...")
    except Exception as e:
        print(f"Error embedding chunk {i}: {e}")

print("Embeddings generated and stored in a simple list.")
print(f"Number of vectors in the store: {len(vector_store)}")

# Function to perform similarity search (simple in-memory implementation)
def retrieve_documents(query_text, k=5):
    query_embedding_response = genai.embed_content(
        model=embedding_model,
        content=query_text,
        task_type="RETRIEVAL_QUERY"
    )
    query_embedding = np.array(query_embedding_response['embedding']).reshape(1, -1)

    # Calculate cosine similarity with all stored document embeddings
    similarities = []
    for doc_embedding, doc_metadata, doc_text in vector_store:
        doc_embedding_np = np.array(doc_embedding).reshape(1, -1)
        similarity = cosine_similarity(query_embedding, doc_embedding_np)[0][0]
        similarities.append((similarity, doc_metadata, doc_text))

    # Sort by similarity and return top k
    similarities.sort(key=lambda x: x[0], reverse=True)
    return similarities[:k]

print("Retrieval function 'retrieve_documents' created.")

Available Embedding Models:
models/gemini-embedding-001
models/gemini-embedding-2-preview
models/gemini-embedding-2
Generating embeddings for document chunks...
Processed 1/386 chunks...
Processed 11/386 chunks...
Processed 21/386 chunks...
Processed 31/386 chunks...
Processed 41/386 chunks...
Processed 51/386 chunks...
Processed 61/386 chunks...
Processed 71/386 chunks...
Processed 81/386 chunks...
Processed 91/386 chunks...
Processed 101/386 chunks...
Processed 111/386 chunks...
Processed 121/386 chunks...
Processed 131/386 chunks...
Processed 141/386 chunks...
Processed 151/386 chunks...
Processed 161/386 chunks...
Processed 171/386 chunks...
Processed 181/386 chunks...
Processed 191/386 chunks...
Processed 201/386 chunks...
Processed 211/386 chunks...
Processed 221/386 chunks...


Error embedding chunk 230: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-2:embedContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 1000, model: gemini-embedding-2
Please retry in 17.597900002s.


Error embedding chunk 234: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-2:embedContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 1000, model: gemini-embedding-2
Please retry in 14.506045453s.


Error embedding chunk 236: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-2:embedContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 1000, model: gemini-embedding-2
Please retry in 13.217879111s.


Error embedding chunk 237: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-2:embedContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 1000, model: gemini-embedding-2
Please retry in 12.91541859s.


...

Error embedding chunk 380: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-2:embedContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 1000, model: gemini-embedding-2
Please retry in 11.981053279s.


Error embedding chunk 381: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-2:embedContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 1000, model: gemini-embedding-2
Please retry in 11.656416904s.


Error embedding chunk 382: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-2:embedContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 1000, model: gemini-embedding-2
Please retry in 11.306820874s.


Error embedding chunk 383: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-2:embedContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 1000, model: gemini-embedding-2
Please retry in 10.890073871s.


Error embedding chunk 384: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-2:embedContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 1000, model: gemini-embedding-2
Please retry in 10.566348573s.
Error embedding chunk 385: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-2:embedContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: gene

## 10. Guardado del Vector Store
Después de generar los embeddings, este bloque guarda el `vector_store` en el archivo especificado (`vector_store.pkl`) usando `pickle`. Esto es útil para persistir el trabajo y evitar la regeneración de embeddings en futuras sesiones, especialmente si el proceso de embedding es costoso o consume cuotas de API.

### Guardar el Vector Store
Una vez que los embeddings se han generado (o se ha resuelto el problema de la cuota), puedes guardar el `vector_store` en un archivo para no tener que volver a generarlo en futuras sesiones.

In [ ]:
# Guardar el vector_store en un archivo usando pickle
try:
    with open(VECTOR_STORE_FILENAME, 'wb') as f:
        pickle.dump(vector_store, f)
    print(f"Vector store guardado exitosamente en '{VECTOR_STORE_FILENAME}'")
except Exception as e:
    print(f"Error al guardar el vector store: {e}")

Vector store guardado exitosamente en './data/vector_store.pkl'


## 11. Interfaz del Chatbot con Gradio
Este es el corazón interactivo del chatbot. Utiliza la librería Gradio para crear una interfaz de chat web. La función `rag_chat` integra la lógica de RAG: recupera documentos relevantes basados en la consulta del usuario, construye un `prompt` contextualizado y lo envía al modelo generativo de Gemini (`gemini-2.5-flash`) para obtener una respuesta. Finalmente, la respuesta incluye las fuentes utilizadas para mayor transparencia. Se incluye un título y una descripción específicos para el chatbot de la Coordinadora Mercantil S.A.

In [ ]:
import google.generativeai as genai
import gradio as gr

# Initialize the Gemini Generative Model
# You can choose a different model if 'gemini-pro' is not available or preferred.
# Ensure 'genai.configure' has been called previously with your API key.
generative_model = genai.GenerativeModel('models/gemini-2.5-flash') # Updated to 'models/gemini-pro-latest'

def rag_chat(query, chat_history):
    # Retrieve relevant documents
    retrieved_results = retrieve_documents(query, k=3) # Retrieve top 3 documents

    # Construct the context for the LLM
    context = ""
    sources = []
    for i, (similarity, metadata, text) in enumerate(retrieved_results):
        context += f"Document {i+1} (Source: {metadata.get('source', 'N/A')}, Page: {metadata.get('page', 'N/A')}, Sheet: {metadata.get('sheet', 'N/A')}):\n{text}\n\n"
        # Ensure we're adding meaningful source info; excel sheets should also be distinct
        source_info = f"Source: {metadata.get('source', 'N/A')}"
        if metadata.get('page') is not None:
            source_info += f", Page: {metadata['page']}"
        if metadata.get('sheet') is not None:
            source_info += f", Sheet: {metadata['sheet']}"
        sources.append(source_info)

    # Prepare the prompt for the generative model
    prompt = f"""You are a helpful assistant. Use the following retrieved information to answer the user's question. If you don't know the answer, just say that you don't know, don't try to make up an answer.

Retrieved Information:
{context}

User Question: {query}

Answer:"""

    # --- Start of modifications ---
    # Clean the prompt string to remove potential invalid characters
    # This helps in case PDF/Excel extraction introduced malformed characters
    prompt = prompt.encode("utf-8", "ignore").decode("utf-8")
    prompt = " ".join(prompt.split()) # Normalize whitespace

    # Debugging: Print prompt length and snippet
    print(f"Debug: Prompt length: {len(prompt)}")
    print(f"Debug: Prompt snippet: {prompt[:500]}...") # Print first 500 chars
    # --- End of modifications ---

    # Generate a response using the Gemini model
    try:
        response = generative_model.generate_content(prompt)
        # Extract text from the response object
        response_text = response.text
    except Exception as e:
        response_text = f"Error generating response: {e}"

    # Append sources to the response
    if sources:
        unique_sources = list(set(sources))
        response_text += "\n\n**Fuentes:**\n" + "\n".join(unique_sources)

    return response_text

# Create the Gradio ChatInterface
print("Launching Gradio interface...")
iface = gr.ChatInterface(
    rag_chat,
    title="Chatbot RAG: Asistente para el Crossorter de Coordinadora Mercantil S.A.", # Updated title
    description=(
        "Este chatbot te ayudará a resolver dudas sobre el plan de mantenimiento de las bandas "
        "transportadoras y la tecnología de clasificación de mercancías del Crossorter."
    ),
    examples=[
        "¿Cuál es el plan de mantenimiento para las bandas transportadoras?", # Added demo question 1
        "¿Qué tecnología utiliza el crossorter para clasificar la mercancía?", # Added demo question 2
        "¿Cual fue el ultimo mantenimiento realizado?"  # Added demo question
    ]
)

# Move the theme argument to the launch method
iface.launch(debug=True, theme=gr.themes.Soft())

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
